In [3]:
import zipfile
zip_path = "/content/calibrated (2).zip"
extract_path = "/content/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted to:", extract_path)

Files extracted to: /content/


In [4]:
import cv2
import numpy as np
import glob
import os
import json
import matplotlib.pyplot as plt


CHECKERBOARD = (7, 9)
SQUARE_SIZE_MM = 20.0
IMAGES_PATH = "/content/calibrated"
OUTPUT_DIR = "/content"
WORK_SCALE = 0.5
ERROR_THRESHOLD = 5.0



def run_calibration():

    print("=" * 60)
    print(" CAMERA CALIBRATION ")
    print("=" * 60)


    criteria = (
        cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
        30,
        0.001
    )

    flags_det = (
        cv2.CALIB_CB_ADAPTIVE_THRESH +
        cv2.CALIB_CB_NORMALIZE_IMAGE
    )


    objp = np.zeros(
        (CHECKERBOARD[0] * CHECKERBOARD[1], 3),
        np.float32
    )

    objp[:, :2] = np.mgrid[
        0:CHECKERBOARD[0],
        0:CHECKERBOARD[1]
    ].T.reshape(-1, 2)

    objp *= SQUARE_SIZE_MM

    objpoints = []
    imgpoints = []
    good_images = []

    images = sorted(
        glob.glob(os.path.join(IMAGES_PATH, "*.jpg")) +
        glob.glob(os.path.join(IMAGES_PATH, "*.jpeg"))
    )

    if len(images) == 0:
        print("ERROR: No images found.")
        return

    print(f"Found {len(images)} images\n")

    for i, fname in enumerate(images):

        img = cv2.imread(fname)

        if img is None:
            print(f"✗ [{i+1:02d}] Cannot read image")
            continue

        h, w = img.shape[:2]

        img_work = cv2.resize(
            img,
            (int(w * WORK_SCALE), int(h * WORK_SCALE))
        )

        gray = cv2.cvtColor(
            img_work,
            cv2.COLOR_BGR2GRAY
        )


        ret, corners = cv2.findChessboardCornersSB(
            gray,
            CHECKERBOARD
        )

        if ret:

            corners = corners.astype(np.float32)

            corners2 = cv2.cornerSubPix(
                gray,
                corners,
                (11, 11),
                (-1, -1),
                criteria
            )

            objpoints.append(objp)
            imgpoints.append(corners2)
            good_images.append(fname)

            print(
                f"✓ [{i+1:02d}] {os.path.basename(fname)}"
            )

        else:

            print(
                f"✗ [{i+1:02d}] "
                f"{os.path.basename(fname)} "
                f"— not detected"
            )

    print(
        f"\nDetected {len(good_images)} / {len(images)} images"
    )

    if len(good_images) < 5:
        print("Too few valid images.")
        return


    print("\nRunning calibration...\n")

    sample = cv2.imread(good_images[0])

    h0, w0 = sample.shape[:2]

    work_size = (
        int(w0 * WORK_SCALE),
        int(h0 * WORK_SCALE)
    )

    ret_err, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
        objpoints,
        imgpoints,
        work_size,
        None,
        None
    )


    mtx_full = mtx.copy()

    mtx_full[0, 0] /= WORK_SCALE
    mtx_full[1, 1] /= WORK_SCALE
    mtx_full[0, 2] /= WORK_SCALE
    mtx_full[1, 2] /= WORK_SCALE


    print("Per-image reprojection errors:\n")

    per_errors = []

    for fname, obj_i, img_i, rvec, tvec in zip(
        good_images,
        objpoints,
        imgpoints,
        rvecs,
        tvecs
    ):

        projected, _ = cv2.projectPoints(
            obj_i,
            rvec,
            tvec,
            mtx,
            dist
        )

        err = float(
            np.sqrt(
                np.mean(
                    (img_i - projected) ** 2
                )
            )
        )

        per_errors.append(err)

        status = "✓" if err < 1.0 else "⚠"

        print(
            f"{status} "
            f"{os.path.basename(fname)}: "
            f"{err:.4f}px"
        )

    print("\nFiltering images...\n")

    filtered_objpoints = []
    filtered_imgpoints = []
    filtered_images = []

    for fname, obj_i, img_i, err in zip(
        good_images,
        objpoints,
        imgpoints,
        per_errors
    ):

        if err < ERROR_THRESHOLD:

            filtered_objpoints.append(obj_i)
            filtered_imgpoints.append(img_i)
            filtered_images.append(fname)

            print(
                f"✓ KEEP "
                f"{os.path.basename(fname)} "
                f"({err:.2f}px)"
            )

        else:

            print(
                f"✗ DROP "
                f"{os.path.basename(fname)} "
                f"({err:.2f}px)"
            )

    print(
        f"\nKept {len(filtered_images)} images"
    )


    if len(filtered_images) >= 5:

        print(
            "\nRecalibrating using filtered images...\n"
        )

        ret_err, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
            filtered_objpoints,
            filtered_imgpoints,
            work_size,
            None,
            None
        )

        objpoints = filtered_objpoints
        imgpoints = filtered_imgpoints
        good_images = filtered_images

        # scale matrix again
        mtx_full = mtx.copy()

        mtx_full[0, 0] /= WORK_SCALE
        mtx_full[1, 1] /= WORK_SCALE
        mtx_full[0, 2] /= WORK_SCALE
        mtx_full[1, 2] /= WORK_SCALE


    print("\n" + "=" * 60)
    print(f"Overall Reprojection Error : {ret_err:.4f}px")

    if ret_err < 0.5:
        rating = "EXCELLENT"
    elif ret_err < 1.0:
        rating = "GOOD"
    elif ret_err < 2.0:
        rating = "ACCEPTABLE"
    else:
        rating = "POOR"

    print(f"Rating : {rating}")
    print("=" * 60)

    print("\nCamera Matrix:")
    print(mtx_full)

    print("\nDistortion Coefficients:")
    print(dist)


    np.save(
        os.path.join(OUTPUT_DIR, "camera_matrix.npy"),
        mtx_full
    )

    np.save(
        os.path.join(OUTPUT_DIR, "dist_coeffs.npy"),
        dist
    )

    calib_data = {
        "reprojection_error": float(ret_err),
        "camera_matrix": mtx_full.tolist(),
        "distortion_coefficients": dist.tolist(),
        "image_size": [w0, h0],
        "work_scale": WORK_SCALE,
        "num_images_used": len(good_images),
        "square_size_mm": SQUARE_SIZE_MM,
        "checkerboard_size": list(CHECKERBOARD)
    }

    with open(
        os.path.join(
            OUTPUT_DIR,
            "calibration_params.json"
        ),
        "w"
    ) as f:

        json.dump(
            calib_data,
            f,
            indent=4
        )


    sample = cv2.imread(good_images[0])

    h, w = sample.shape[:2]

    new_mtx, roi = cv2.getOptimalNewCameraMatrix(
        mtx_full,
        dist,
        (w, h),
        1,
        (w, h)
    )

    undistorted = cv2.undistort(
        sample,
        mtx_full,
        dist,
        None,
        new_mtx
    )

    x, y, rw, rh = roi

    undistorted_crop = undistorted[
        y:y+rh,
        x:x+rw
    ]

    fig, ax = plt.subplots(
        1,
        2,
        figsize=(16, 8)
    )

    ax[0].imshow(
        cv2.cvtColor(
            sample,
            cv2.COLOR_BGR2RGB
        )
    )
    ax[0].set_title("Original")
    ax[0].axis("off")

    ax[1].imshow(
        cv2.cvtColor(
            undistorted_crop,
            cv2.COLOR_BGR2RGB
        )
    )
    ax[1].set_title("Undistorted")
    ax[1].axis("off")

    plt.suptitle(
        f"Reprojection Error: {ret_err:.4f}px"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            "undistortion_comparison.png"
        ),
        dpi=150
    )

    plt.close()

    print("\nSaved:")
    print(" camera_matrix.npy")
    print(" dist_coeffs.npy")
    print(" calibration_params.json")
    print(" undistortion_comparison.png")

    print("\n CALIBRATION COMPLETE")

    return mtx_full, dist, ret_err


if __name__ == "__main__":
    run_calibration()

 CAMERA CALIBRATION 
Found 39 images

✓ [01] IMG_20260613_151813_463.jpg
✓ [02] IMG_20260613_151822_285.jpg
✗ [03] IMG_20260613_151828_820.jpg — not detected
✓ [04] IMG_20260613_151919_364.jpg
✓ [05] IMG_20260613_151921_884.jpg
✓ [06] IMG_20260613_151937_251.jpg
✓ [07] IMG_20260613_151939_432.jpg
✓ [08] IMG_20260613_151941_567.jpg
✗ [09] IMG_20260613_151958_842.jpg — not detected
✓ [10] IMG_20260613_152049_300.jpg
✗ [11] IMG_20260613_152052_977.jpg — not detected
✓ [12] IMG_20260613_152104_747.jpg
✗ [13] IMG_20260613_152120_389.jpg — not detected
✗ [14] IMG_20260613_152123_513.jpg — not detected
✓ [15] WhatsApp Image 2026-06-13 at 4.24.35 PM.jpeg
✓ [16] WhatsApp Image 2026-06-13 at 4.24.36 PM (1).jpeg
✓ [17] WhatsApp Image 2026-06-13 at 4.24.36 PM (2).jpeg
✓ [18] WhatsApp Image 2026-06-13 at 4.24.36 PM.jpeg
✓ [19] WhatsApp Image 2026-06-13 at 4.24.37 PM (1).jpeg
✓ [20] WhatsApp Image 2026-06-13 at 4.24.37 PM (2).jpeg
✓ [21] WhatsApp Image 2026-06-13 at 4.24.37 PM.jpeg
✓ [22] WhatsApp I